# Public Narrative Synthesis Dataset

In [ ]:
import json
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from newspaper import Article, Config

# ── Configuration ─────────────────────────────────────────────────────────────
YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TOP_N = 20
DELAY = 1.5

WIKI_HEADERS = {"User-Agent": "NewsTopicResearcher/1.0 (research project; contact@example.com)"}
WIKI_BASE = "https://en.wikipedia.org/wiki/{year}_in_the_United_States"

OUTPUT_JSON = "wikipedia_top_news_us.json"
OUTPUT_CSV = "wikipedia_top_news_us.csv"


def get_reference_links(soup, ref_number):
    ref_id = f"cite_note-{ref_number}"
    ref_item = soup.find("li", id=ref_id)

    if not ref_item:
        return []

    links = []
    seen = set()

    for a in ref_item.find_all("a", href=True):
        href = a["href"]
        if href.startswith("http") and "web.archive.org" not in href and href not in seen:
            seen.add(href)
            links.append(href)

    return links


def fetch_article_text(url):
    config = Config()
    config.browser_user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    config.request_timeout = 10

    try:
        article = Article(url, config=config)
        article.download()
        article.parse()
        return article.text
    except Exception:
        print(f"      [Timeout/Error] Skipping {url[:50]}...")
        return None


def clean_text_and_extract_date(text):
    date_match = re.search(
        r"(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2}",
        text
    )
    date = date_match.group() if date_match else None

    refs = re.findall(r"\[\s*(\d+)\s*\]", text)

    text = re.sub(
        r"^(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2}\s*([–\-:])?\s*",
        "",
        text
    )
    text = re.sub(r"\[\s*\d+\s*\]", "", text)

    return date, text.strip(), refs


def process_top_n_events(events, soup, limit=10, min_sources=2, preferred_sources=3):
    selected_events = []

    print(f"\n--- Starting Deep Scrape (Target: {limit} successful stories) ---")

    for event in events:
        if len(selected_events) >= limit:
            break

        print(f"Checking Candidate: {event['event'][:80]}... (Current Selected: {len(selected_events)})")
        print("Refs:", event.get("refs", []))

        candidate_links = []
        seen_links = set()

        # Collect unique candidate links for this event
        for ref in event.get("refs", []):
            ref_links = get_reference_links(soup, ref)
            for ref_link in ref_links:
                if ref_link not in seen_links:
                    seen_links.add(ref_link)
                    candidate_links.append(ref_link)

        # Skip early if not enough candidate links
        if len(candidate_links) < min_sources:
            print(f"   [Skipped Early] Only {len(candidate_links)} candidate link(s)")
            continue

        collected_sources = []

        for ref_link in candidate_links:
            content = fetch_article_text(ref_link)

            if content and len(content) > 500:
                collected_sources.append({
                    "source_link": ref_link,
                    "full_source_text": content
                })
                print(f"   [Success] Scraped news link {len(collected_sources)}: {ref_link[:60]}...")

                if len(collected_sources) >= preferred_sources:
                    break

        # Keep only events with enough usable articles
        if len(collected_sources) >= min_sources:
            event["source_articles"] = collected_sources
            event["num_source_articles"] = len(collected_sources)

            for i in range(3):
                if i < len(collected_sources):
                    event[f"source_link_{i+1}"] = collected_sources[i]["source_link"]
                    event[f"full_source_text_{i+1}"] = collected_sources[i]["full_source_text"]
                else:
                    event[f"source_link_{i+1}"] = None
                    event[f"full_source_text_{i+1}"] = None

            selected_events.append(event)
        else:
            print(f"   [Skipped] Only found {len(collected_sources)} usable source article(s)")

    # Prioritize events with 3 sources over those with 2
    selected_events = sorted(
        selected_events,
        key=lambda x: x.get("num_source_articles", 0) >= preferred_sources,
        reverse=True
    )

    return selected_events[:limit]


def get_direct_text(tag):
    parts = []
    for child in tag.contents:
        
        if getattr(child, "name", None) in ("ul", "ol", "dl"):
            continue

        if isinstance(child, str):
            text = child.strip()
        else:
            text = child.get_text(" ", strip=True)

        if text:
            parts.append(text)

    return " ".join(parts).strip()

def fetch_year_page_events(year: int):
    url = WIKI_BASE.format(year=year)
    print(f"  Fetching: {url}")

    try:
        resp = requests.get(url, headers=WIKI_HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  [ERROR] Could not fetch {url}: {e}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    content = soup.find("div", {"class": "mw-parser-output"})
    if not content:
        print(f"  [ERROR] Could not find main content for {year}")
        return []

    events = []
    source_url = f"https://en.wikipedia.org/wiki/{year}_in_the_United_States"

    month_names = {
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"
    }

    in_events_section = False
    current_month = None

    # walk the page in order
    for tag in content.find_all(["h2", "h3", "ul", "ol", "dl"]):
        if tag.name == "h2":
            heading_text = tag.get_text(" ", strip=True).replace("[edit]", "").strip()

            if heading_text == "Events":
                in_events_section = True
                current_month = None
                continue

            if in_events_section:
                break

        elif tag.name == "h3":
            if not in_events_section:
                continue

            heading_text = tag.get_text(" ", strip=True).replace("[edit]", "").strip()
            if heading_text in month_names:
                current_month = heading_text
            else:
                current_month = None
            continue

        # only process lists when we are inside Events and inside a month
        if not in_events_section or not current_month:
            continue

        # skip nested lists completely
        if tag.find_parent(["li", "dd"]) is not None:
            continue

        # process only top-level bullets in this list
        for item in tag.find_all(["li", "dd"], recursive=False):
            # if this bullet contains nested bullets, skip the whole bullet
            if item.find(["ul", "ol", "dl"], recursive=False) is not None:
                continue

            text = get_direct_text(item)

            if len(text) < 30:
                continue

            date, clean_text, refs = clean_text_and_extract_date(text)

            events.append({
                "year": year,
                "month": current_month,
                "date": date,
                "event": clean_text,
                "source": source_url,
                "refs": refs
            })

    print(f"  Found {len(events)} event bullets in {year}")
    return events


def save_results(results):
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\nJSON saved → {OUTPUT_JSON}")

    flat = [event for year_events in results.values() for event in year_events]
    if flat:
        df = pd.DataFrame(flat)[[
            "year", "month", "date", "event", "source", "refs",
            "num_source_articles",
            "source_link_1", "full_source_text_1",
            "source_link_2", "full_source_text_2",
            "source_link_3", "full_source_text_3"
        ]]
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"CSV saved → {OUTPUT_CSV}")


def main():
    print("=" * 65)
    print("  Script 1: Wikipedia Top News Extractor")
    print(f"  Years : {YEARS[0]} – {YEARS[-1]}")
    print(f"  Top N : {TOP_N}")
    print("=" * 65)

    all_results = {}

    for year in YEARS:
        print(f"\n── {year} ──────────────────────────────────")

        events = fetch_year_page_events(year)
        time.sleep(DELAY)

        if not events:
            all_results[year] = []
            continue

        html = requests.get(
            f"https://en.wikipedia.org/wiki/{year}_in_the_United_States",
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=30
        ).text
        soup = BeautifulSoup(html, "html.parser")

        filtered_events = process_top_n_events(
            events,
            soup,
            limit=TOP_N,
            min_sources=2,
            preferred_sources=3
        )
        all_results[year] = filtered_events

        print(f"  ✓ Top {TOP_N} events identified for {year}")
        time.sleep(DELAY)

    save_results(all_results)
    return all_results


if __name__ == "__main__":
    main()

  Script 1: Wikipedia Top News Extractor
  Years : 2020 – 2025
  Top N : 20

── 2020 ──────────────────────────────────
  Fetching: https://en.wikipedia.org/wiki/2020_in_the_United_States
  Found 121 event bullets in 2020

--- Starting Deep Scrape (Target: 20 successful stories) ---
Checking Candidate: 2019–2021 Persian Gulf crisis: President Donald Trump approves the targeted kill... (Current Selected: 0)
Refs: ['7']
   [Skipped Early] Only 1 candidate link(s)
Checking Candidate: The 77th Golden Globe Awards are held in Beverly Hills, California .... (Current Selected: 0)
Refs: ['8']
   [Skipped Early] Only 1 candidate link(s)
Checking Candidate: Former film producer Harvey Weinstein is charged with four additional counts of ... (Current Selected: 0)
Refs: ['9']
   [Skipped Early] Only 1 candidate link(s)
Checking Candidate: President Donald Trump and China's Vice Premier Liu He sign the U.S.–China Phase... (Current Selected: 0)
Refs: ['17', '18']
   [Success] Scraped news link 1: htt

In [2]:
df=pd.read_csv("wikipedia_top_news_us.csv")
len(df)

55

## Constraint-based ground truth generation

In [10]:
import pandas as pd

developer_prompt = """
You are an expert news analysis assistant. Your task is to read {num_of_articles} reference news articles provided by the user about the same event and produce ONE unified constraint-based ground truth for news article writing.

You will be given:
1. The event title
2. {num_of_articles} source news articles about the same event

Your goal is NOT to write the final article.
Your goal is to combine the {num_of_articles} articles into a single structured ground truth that captures:
- the core event
- the important entities
- the key information that should be included
- the expected article structure
- the important statistics or claims
- any disagreements or uncertainties across sources

You must merge the {num_of_articles} articles into one consolidated representation. Do NOT keep the sources separate in the final output. Do NOT organize the output by source IDs.

========================
TASK
========================

Step 1: Read all {num_of_articles} articles and identify the shared and important information.

From the {num_of_articles} articles together, determine:

1. Event Type
- What kind of event is this?
Examples: political decision, military strike, natural disaster, court ruling, protest, accident, diplomatic meeting, election result, etc.

2. Core Entities
Identify the most important entities across the {num_of_articles} articles:
- people
- organizations
- locations
- other important named entities

3. Key Information
Synthesize the important information under these categories:
- what happened
- who is involved
- why it matters
- possible impact

4. Common Article Structure
Infer the most likely shared structure across the {num_of_articles} articles, such as:
- lead
- important details
- background/context
- implications/next steps

5. Statistics and Specific Claims
Extract important:
- numbers
- dates
- counts
- financial values
- percentages
- official statements
- factual claims

When sources agree, treat the information as strong ground truth.
When sources partially overlap, keep the shared core.
When sources disagree, do NOT invent a resolution. Instead, explicitly mark the point as disputed or variable.

========================
GROUND TRUTH CONSTRUCTION RULES
========================

Create ONE consolidated ground truth with these principles:

- Keep only information supported by the provided articles.
- Prefer information that appears consistently across multiple articles.
- Merge overlapping information into one concise constraint.
- Do not duplicate the same fact in slightly different wording.
- If a detail appears in only one source and is not central, mark it as optional or uncertain rather than mandatory.
- If statistics or claims conflict across sources, preserve the disagreement explicitly.
- Do not add outside knowledge.
- Do not write a full article.
- Produce evaluable constraints, not prose narration.

========================
OUTPUT FORMAT
========================

Only return the result in valid JSON with this exact structure:

{{
  "event_title": "...",
  "event_type": "...",
  "ground_truth": {{
    "core_event_summary": "...",
    "entities": {{
      "required": [],
      "optional": []
    }},
    "key_information": {{
      "what_happened": [],
      "who_is_involved": [],
      "why_it_matters": [],
      "possible_impact": []
    }},
    "structural_expectations": {{
      "lead": [],
      "details": [],
      "background_context": [],
      "ending_or_forward_look": []
    }},
    "statistics_and_claims": {{
      "agreed_upon": [],
      "disputed_or_variable": []
    }}
  }}
}}
"""

user_prompt = """Here is the event title:
{EVENT_TITLE}

Here are the reference articles:

{ARTICLES_INPUT}
"""


def build_article_input(articles):
    parts = []
    for i, article in enumerate(articles, start=1):
        parts.append(f"Article {i}:\n{article}")
    return "\n\n".join(parts)


def collect_articles_from_row(row):
    articles = []
    for i in [1, 2, 3]:
        col = f"full_source_text_{i}"
        if col in row and pd.notna(row[col]) and str(row[col]).strip() != "":
            articles.append(str(row[col]).strip())
    return articles
  
def build_user_prompt(event_title, articles_input):
    return user_prompt.format(
            EVENT_TITLE=event_title,
            ARTICLES_INPUT=articles_input
        )

def build_developer_and_user_prompt(row):
    event_title = str(row["event"]).strip()
    articles = collect_articles_from_row(row)

    if len(articles) < 2:
        raise ValueError("Each row must have at least 2 source articles.")

    articles_input = build_article_input(articles)

    final_developer_prompt = developer_prompt.format(
        num_of_articles=len(articles)
    )
    return final_developer_prompt, build_user_prompt(event_title, articles_input)
  
def call_gpt5_mini(developer_prompt, user_prompt):
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": developer_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_completion_tokens=10000,
        n=1
    )
    return response.choices[0].message.content

In [ ]:
import json
import openai

client = openai.OpenAI(api_key="Paste your key here")  

SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "event_title": {"type": "string"},
        "event_type": {"type": "string"},
        "ground_truth": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "core_event_summary": {"type": "string"},
                "entities": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "required": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "optional": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": ["required", "optional"]
                },
                "key_information": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "what_happened": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "who_is_involved": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "why_it_matters": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "possible_impact": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": [
                        "what_happened",
                        "who_is_involved",
                        "why_it_matters",
                        "possible_impact"
                    ]
                },
                "structural_expectations": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "lead": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "details": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "background_context": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "ending_or_forward_look": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": [
                        "lead",
                        "details",
                        "background_context",
                        "ending_or_forward_look"
                    ]
                },
                "statistics_and_claims": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "agreed_upon": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "disputed_or_variable": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": ["agreed_upon", "disputed_or_variable"]
                }
            },
            "required": [
                "core_event_summary",
                "entities",
                "key_information",
                "structural_expectations",
                "statistics_and_claims"
            ]
        }
    },
    "required": ["event_title", "event_type", "ground_truth"]
}


def call_gpt5_mini_with_schema(developer_prompt, user_prompt):
    try:
        response = client.responses.create(
            model="gpt-5-mini",
            instructions=developer_prompt,
            input=user_prompt,
            store=False,
            text={
                "format": {
                    "type": "json_schema",
                    "name": "constraint_ground_truth",
                    "schema": SCHEMA,
                    "strict": True
                }
            }
        )

        parsed = json.loads(response.output_text)
        return parsed, None

    except Exception as e:
        return None, str(e)



In [31]:
df = pd.read_csv("wikipedia_top_news_us.csv")

In [34]:
len(df)

55

In [35]:
results = []
errors = []

for idx, row in df.iterrows():
    try:
        dev_prompt, usr_prompt = build_developer_and_user_prompt(row)
        parsed, err = call_gpt5_mini_with_schema(dev_prompt, usr_prompt)

        if err is not None:
            errors.append({"row_id": idx, "event": row["event"], "error": err})
            continue

        results.append({
            "row_id": idx,
            "year": row["year"],
            "month": row["month"],
            "date": row["date"],
            "event": row["event"],
            "ground_truth_json": json.dumps(parsed, ensure_ascii=False)
        })

    except Exception as e:
        errors.append({"row_id": idx, "event": row.get("event", ""), "error": str(e)})

results_df = pd.DataFrame(results)
errors_df = pd.DataFrame(errors)

results_df.to_csv("ground_truth_results.csv", index=False, encoding="utf-8")
errors_df.to_csv("ground_truth_errors.csv", index=False, encoding="utf-8")

In [37]:
len(errors_df)

0

In [38]:
len(results_df)

55